# Clase 6: programa tu propio KNN y compáralo con scikit-learn

En la clase anterior usaste `KNeighborsClassifier` de scikit-learn para clasificar tumores del conjunto de datos de cáncer de mama. Con pocas líneas obtuviste un modelo que funcionaba, pero, ¿sabes realmente qué pasa dentro de esa función cuando le pasas un punto nuevo y te devuelve una predicción?

Hoy abres la caja negra. Vas a programar tu propio clasificador de los k vecinos más cercanos (KNN, por sus siglas en inglés *K-Nearest Neighbors*) usando solo Python y NumPy, sin scikit-learn. No se trata de sustituir la librería, que está mucho más optimizada que cualquier cosa que escribas en una clase, sino de entender el mecanismo exacto que hay detrás. Esa comprensión te ayuda a depurar modelos, elegir mejor los hiperparámetros y reconocer cuándo un algoritmo no es la herramienta adecuada para tu problema.

Al final de la clase comparas tu implementación con la de scikit-learn sobre el mismo conjunto de datos y compruebas que ambas llegan a resultados muy parecidos. Todo el código de este cuaderno funciona directamente en Google Colab, sin conexiones externas: solo necesitas NumPy y scikit-learn, que ya vienen instalados.

## El algoritmo paso a paso

¿Cómo eliges un restaurante en una ciudad que no conoces? Seguramente no preguntas a todos los habitantes qué les parece cada local. Preguntas a las personas que tienes más cerca, quizás las cinco o diez que viven en tu mismo barrio, y te fías de lo que opina la mayoría de ese grupo reducido. Eso es exactamente lo que hace KNN con los datos.

El proceso se reduce a tres acciones concretas:

1. Calcular la distancia entre el punto nuevo y cada uno de los puntos del conjunto de entrenamiento
2. Ordenar esas distancias de menor a mayor y quedarte con las k más pequeñas, es decir, con los k vecinos más próximos
3. Votar: la clase que más se repite entre esos k vecinos se convierte en la predicción para el punto nuevo

¿Ves alguna fase de entrenamiento que ajuste parámetros internos, como ocurre en la regresión lineal? No la hay. KNN guarda el conjunto de entrenamiento tal cual y hace todo el trabajo en el momento de predecir. Por eso se le llama un aprendiz perezoso (*lazy learner*): no aprende un modelo compacto, simplemente compara.

## Programa tu propio clasificador KNN

Empieza por la pieza más pequeña: una función que calcule la distancia euclídea entre dos puntos, es decir, la distancia en línea recta entre ambos. NumPy te permite hacerlo en una sola línea gracias a `numpy.linalg.norm`, que calcula la longitud del vector diferencia entre los dos puntos.

Con esa pieza ya puedes construir la función principal, `knn_predecir`, que recibe el conjunto de entrenamiento, el punto nuevo y el valor de k y sigue exactamente los tres pasos que acabas de leer.

In [ ]:
import numpy as np
from collections import Counter


def distancia_euclidea(punto_a, punto_b):
    """Distancia en línea recta entre dos puntos."""
    punto_a = np.array(punto_a)
    punto_b = np.array(punto_b)
    return np.linalg.norm(punto_a - punto_b)


def knn_predecir(datos_entrenamiento, punto_nuevo, k):
    """
    datos_entrenamiento: lista de tuplas (caracteristicas, etiqueta)
    punto_nuevo: array con las caracteristicas del punto a clasificar
    k: numero de vecinos que participan en la votacion
    """
    if len(datos_entrenamiento) < k:
        raise ValueError("k no puede superar el número de puntos de entrenamiento")

    distancias = []
    for caracteristicas, etiqueta in datos_entrenamiento:
        distancia = distancia_euclidea(caracteristicas, punto_nuevo)
        distancias.append((distancia, etiqueta))

    # Paso 2: ordenar y quedarte con los k vecinos más cercanos
    distancias.sort(key=lambda par: par[0])
    k_vecinos = distancias[:k]

    # Paso 3: votar la clase mayoritaria
    votos = [etiqueta for _, etiqueta in k_vecinos]
    clase_ganadora = Counter(votos).most_common(1)[0][0]
    return clase_ganadora

Fíjate en tres detalles del código:

- `datos_entrenamiento` es una lista de tuplas que empareja cada vector de características con su etiqueta, así la función conoce a la vez dónde está cada punto y a qué clase pertenece
- La comprobación inicial evita un error habitual: pedir más vecinos de los que existen en el conjunto de entrenamiento
- `Counter.most_common(1)` te devuelve, en una sola llamada, la etiqueta que aparece con más frecuencia entre los votos

Con esto ya tienes un clasificador KNN completo. Es más lento que el de scikit-learn porque recorre la lista con un bucle explícito en Python en lugar de usar operaciones vectorizadas internas en C, pero hace exactamente lo mismo a nivel conceptual.

### Compruébalo con un ejemplo diminuto

Antes de lanzarte a un conjunto de datos real, comprueba que la función hace lo que esperas con cuatro puntos inventados: dos etiquetados como "manzana" y dos como "pera". El punto de prueba está mucho más cerca del grupo de las manzanas, así que la predicción con un vecino (k = 1) debería devolver "manzana".

In [ ]:
# Cuatro puntos de entrenamiento inventados, cada uno con su etiqueta
datos_juguete = [
    (np.array([1.0, 2.0]), "manzana"),
    (np.array([1.5, 1.8]), "manzana"),
    (np.array([5.0, 8.0]), "pera"),
    (np.array([6.0, 9.0]), "pera"),
]

punto_prueba = np.array([1.2, 1.9])
print("Predicción para el punto de prueba:", knn_predecir(datos_juguete, punto_prueba, k=1))

## Compara tu implementación con la de scikit-learn

Toca la prueba de fuego: aplicar `knn_predecir` sobre el mismo conjunto de datos de cáncer de mama que usaste en la clase anterior y comprobar si el porcentaje de aciertos se acerca al de `KNeighborsClassifier`.

Como KNN se basa en distancias, y algunas características de este conjunto de datos tienen escalas muy distintas entre sí, escala primero los datos con `StandardScaler`. Así ninguna característica domina el cálculo de distancia solo por tener valores más grandes.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Cargamos el mismo conjunto de datos que en la clase anterior
datos = load_breast_cancer()
X, y = datos.data, datos.target

# Escalamos las características para que todas pesen lo mismo en la distancia
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)

# Dividimos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_escalado, y, test_size=0.2, random_state=42
)

# Preparamos los datos en el formato que espera tu función
datos_entrenamiento = list(zip(X_train, y_train))

k = 5

# Predicciones con tu propio KNN
predicciones_propias = [
    knn_predecir(datos_entrenamiento, punto, k) for punto in X_test
]
precision_propia = accuracy_score(y_test, predicciones_propias)

# Predicciones con scikit-learn
modelo_sklearn = KNeighborsClassifier(n_neighbors=k)
modelo_sklearn.fit(X_train, y_train)
predicciones_sklearn = modelo_sklearn.predict(X_test)
precision_sklearn = accuracy_score(y_test, predicciones_sklearn)

print(f"Precisión de tu KNN:        {precision_propia:.3f}")
print(f"Precisión de scikit-learn:  {precision_sklearn:.3f}")

Si ejecutas este código, ves que ambas precisiones rondan entre el 95 % y el 97 %, con diferencias mínimas entre una y otra. Eso confirma algo importante: tu implementación no es una simplificación aproximada, es el mismo algoritmo, solo que sin las optimizaciones internas de scikit-learn. Las pequeñas diferencias que puedas observar suelen deberse a cómo cada implementación resuelve los empates en la votación, no a un error de concepto.

**Prueba ahora con distintos valores de k y observa qué pasa con la precisión.** La siguiente celda repite la comparación con k = 3, 5, 7 y 11.

In [ ]:
for k_prueba in [3, 5, 7, 11]:
    predicciones_propias = [
        knn_predecir(datos_entrenamiento, punto, k_prueba) for punto in X_test
    ]
    precision_propia = accuracy_score(y_test, predicciones_propias)

    modelo_sklearn_k = KNeighborsClassifier(n_neighbors=k_prueba)
    modelo_sklearn_k.fit(X_train, y_train)
    precision_sklearn_k = accuracy_score(y_test, modelo_sklearn_k.predict(X_test))

    print(f"k = {k_prueba:>2}: tu KNN = {precision_propia:.3f}, scikit-learn = {precision_sklearn_k:.3f}")

¿Notas que ni tu función ni la de scikit-learn mejoran de forma constante al aumentar k? A partir de cierto punto, un k demasiado grande empieza a diluir la información local y la precisión se estanca o incluso baja. Elegir k no es "cuanto más, mejor": es encontrar el punto en el que el vecindario es lo bastante grande para filtrar ruido, pero lo bastante pequeño para seguir siendo local.

## Mide el coste computacional con `time`

Todo lo anterior habla de precisión, pero KNN tiene otro coste que no aparece en esa métrica: el tiempo. Cada predicción obliga a calcular la distancia entre el punto nuevo y todos los puntos del conjunto de entrenamiento, así que, ¿qué pasa con el tiempo de ejecución a medida que ese conjunto crece? Compruébalo con el módulo `time`, comparando tu función con `KNeighborsClassifier` para varios tamaños del conjunto de entrenamiento.

In [ ]:
import time

tamanos = [50, 100, 200, 400]

for tamano in tamanos:
    subconjunto = datos_entrenamiento[:tamano]

    inicio = time.perf_counter()
    for punto in X_test:
        knn_predecir(subconjunto, punto, k)
    tiempo_propio = time.perf_counter() - inicio

    modelo_sklearn.fit(X_train[:tamano], y_train[:tamano])
    inicio = time.perf_counter()
    modelo_sklearn.predict(X_test)
    tiempo_sklearn = time.perf_counter() - inicio

    print(
        f"Tamaño {tamano}: tu función {tiempo_propio:.4f} s, "
        f"scikit-learn {tiempo_sklearn:.4f} s"
    )

¿Ves cómo crece el tiempo de tu función mucho más rápido que el de scikit-learn a medida que aumenta el tamaño del conjunto de entrenamiento? Esa diferencia es la prueba práctica del coste computacional de KNN: la librería usa estructuras de datos y optimizaciones internas que tu versión, escrita para aprender el algoritmo, no incluye.

## Ventajas y límites de KNN

Programar el algoritmo desde cero deja mucho más claros sus puntos fuertes y débiles que usarlo como caja cerrada.

**Coste computacional.** Cada predicción obliga a calcular la distancia entre el punto nuevo y todos los puntos del conjunto de entrenamiento. Si tienes 100 puntos, son 100 cálculos de distancia; si tienes 1.000.000, son 1.000.000, y eso solo para clasificar un único punto nuevo. Como no existe una fase de entrenamiento que resuma la información en un modelo compacto, KNN no escala bien cuando el volumen de datos crece mucho, y cada consulta se vuelve más lenta a medida que añades más ejemplos.

**Sensibilidad a la escala de las características.** El algoritmo asume que todas las dimensiones aportan por igual a la distancia. Si una característica se mide en miles y otra en decimales, la primera domina el cálculo aunque no sea más relevante para el problema. Por eso escalaste los datos antes de comparar tu función con scikit-learn: sin ese paso, la comparación habría sido injusta y los resultados, peores de lo esperado.

**Cuándo evitar KNN.** Conviene descartarlo, o al menos usarlo con cautela, en estos casos:

- Conjuntos de datos muy grandes donde necesitas predicciones rápidas y constantes en producción
- Espacios con muchas dimensiones, donde el concepto de "cercanía" pierde significado porque casi todos los puntos acaban a distancias parecidas entre sí
- Clases muy desequilibradas, donde una clase mayoritaria puede dominar la votación aunque el punto nuevo se parezca más a la clase minoritaria
- Datos con muchas características irrelevantes, que añaden ruido a la distancia sin aportar información útil

A cambio, KNN sigue siendo una opción razonable cuando el conjunto de datos es pequeño o mediano, las clases están razonablemente equilibradas y quieres un modelo fácil de explicar a alguien sin formación técnica.

## Resumen: lo que te llevas de esta clase

- Programar KNN desde cero se reduce a tres pasos: calcular distancias, ordenarlas y votar la clase mayoritaria entre los k vecinos más próximos
- La función `knn_predecir` necesita el conjunto de entrenamiento con sus etiquetas, el punto nuevo y el valor de k
- NumPy simplifica el cálculo de la distancia euclídea con `numpy.linalg.norm`, evitando bucles manuales sobre cada dimensión
- Tu implementación y la de scikit-learn logran precisiones muy similares sobre el mismo conjunto de datos, lo que confirma que ambas aplican el mismo principio
- KNN es un aprendiz perezoso: no construye un modelo compacto, compara el punto nuevo contra todo el conjunto de entrenamiento en cada predicción
- El coste computacional y la sensibilidad a la escala de las características son las dos limitaciones que más debes vigilar

**Ahora te toca a ti.** Abre el cuaderno de práctica de esta clase y programa tu propio KNN sobre un conjunto de datos distinto, el de los vinos.

**Siglas usadas en esta clase:** KNN (K-Nearest Neighbors, k vecinos más cercanos).